# Dataset preprocessing

Build the binary jailbreak / non-jailbreak tables from the raw prompt files.

The split is stratified and seeded. Prompts are deduplicated after the same
cleaner is applied to both classes, and synonym augmentation is dropped when it
collides with a training or test prompt. This notebook writes
`data/binary_data/training_data_binary.csv` and
`data/binary_data/testing_data_binary.csv`. It does not modify the raw source
CSVs.

Running it replaces the checked-in binary files, which have already been
deduplicated in place.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import pandas as pd

from binary_data import (
    build_binary_dataset,
    default_text_preprocess,
    seeded_synonym_replacement,
    write_binary_splits,
)
from paths import data_path

jailbreak_df = pd.read_csv(data_path("jailbreak_prompts_DAN.csv"))
regular_df = pd.read_csv(data_path("regular_prompts_DAN.csv"))
train_df, test_df = build_binary_dataset(
    jailbreak_df,
    regular_df,
    preprocess_fn=default_text_preprocess,
    augment_fn=seeded_synonym_replacement(42),
    regular_cap=2000,
    test_size=0.2,
    random_state=42,
)
train_path, test_path = write_binary_splits(train_df, test_df)
print(f"Wrote {len(train_df)} training rows to {train_path}")
print(f"Wrote {len(test_df)} test rows to {test_path}")
print("Raw source CSVs were not modified.")